# Erste Bayessche Auswertung
### Analyse der Unsicherheit zu Fragen mit zwei Antwortmöglichkeiten (z.B. Ja/Nein).

Bleiben wir bei dem Beispiel zum
Event im Verein. Wir hätten die Umfrage Teilnehmer:innen gefragt ob sie an unserer
nächsten Veranstaltung teilnehmen wollen. Wir haben 20 zufällig ausgewählte Mitlgieder
befragt und 18/20 Teilnehmer:innen antworten, dass sie gern dabei seien wollen. Was
sagt uns das darüber aus welche Anteile an der Gesamtheit der (z.B. 1000) Mitglieder
plausibel dabei sein wollen?

Wir können an diesem Beispiel in der Praxis sehen wie man von einer Prior Verteilung mit Daten zu einer Posterior Verteilung kommt.

In [ ]:
import arviz as az
import bambi as bmb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import expit

In [ ]:
# Beispiel-Daten: Umfrage bei einem Verein "Würden Sie am Sommerfest
# teilnehmen?"
np.random.seed(42)

n_befragte = 20
n_ja = 18

In [ ]:
# Daten als DataFrame (Bambi bevorzugt DataFrames)
antworten = np.concatenate([np.ones(n_ja), np.zeros(n_befragte - n_ja)])
df = pd.DataFrame(
    {
        "teilnahme": antworten  # 1 = Ja, 0 = Nein
    }
)

print(f"Umfrage-Ergebnis: {n_ja} von {n_befragte} würden teilnehmen")
print(f"Beobachteter Anteil: {n_ja / n_befragte:.1%}\n")

### Bambi
Wir nutzen für Bayessche Statistik hier das Package Bambi. Bambi ermöglicht es statistische Modelle mit einem String zu spezifizieren. Unsere Daten bestehen aus der binären Variable "teilnahme" und wir wollen herausfinden welche Anteile an Vereinsmitgliedern realistischerweise Ja sagen würden wenn wir statt den 20 alle 1000 befragen würden. Das ist aus mathematischer Sicht vergleichbar mit der Tendenz einer Münze Kopf zu zeigen. Wir suchen also wie im Eingangsbeispiel nach einer Tendenz oder einem Anteil. Anteile liegen wieder zwischen 0 und 1.

Nun fehlt uns noch ein statistisches Modell um zu beschreiben wie wir uns vereinfacht vorstellen wie die Daten, die wir gesammelt haben generiert wurden. Nahezu alle statistischen Verfahren, ob klassisch oder Bayessch treffen solche Modellannahmen. In der Bayesschen Statistik ist es sehr typisch diese Annahmen transparent darzustellen. Wir wollen also die Tendenz zur Teilnahme modellieren. Wir nutzen hier keine zusätzliche Vorhersagevariable (wie z.B. das Alter) sondern modellieren einfach die durchschnittliche Tendenz für alle Vereinsmitglieder. 

In [ ]:
# In Bambi können wir dieses Modell einfach mit "teilnahme ~ 1" beschreiben. (Das Argument family=bernoulli erstmal.)
# Das heißt übersetzt: "Wir modellieren die Wahrscheinlichkeit zur Teilnahme ohne zusätzliche Vorhersagevariablen.
model = bmb.Model("teilnahme ~ 1", df, family="bernoulli")
model.build()


Damit wir die statistische Auswertung machen können, müssen wir einen Prior Wissensstand definieren. Zum Glück benutzt Bambi per default gute "weakly infomative priors", die sich als Standard etabliert haben. Das heißt für uns ganz praktisch, dass wir uns nicht damit befassen müssen einen Prior zu definieren. Ganz am Ende des Notebooks zeigen wir euch einmal wie man sich den Prior anschauen könnte. Das ist aber für euch optional.

Als nächstes wollen wir den Prior mit unseren Daten (18/20 Ja Antworten) kombinieren, um den Posterior zu bekommen. Der Posterior beschreibt unseren (Un)wissensstand zur Teilnahme Tendenz der Mitglieder nachdem wir die Daten gesehen haben.

In der Bayesschen Statistik nutzt man (nahezu) immer ein numerisches Verfahren um an den Posterior zu kommen. Es heißt Markov Chain Monte Carlo (MCMC) und erlaubt es uns die Posterior Verteilung approximativ darzustellen.

In Bambi ist das ganz leicht. Wir müssen nur die Funktion model.fit() ausführen.

In [ ]:
results = model.fit(draws=5000, chains=4, random_seed=42)

Nachdem Bambi MCMC genutzt hat um den Posterior zu approximieren muss man einmal prüfen, dass der R-Hat Wert nicht über 1.01 liegt. R-Hat ist ein Wert, der überprüft ob die numerische Approximation gut geklappt hat.

In [ ]:
az.rhat(results)["Intercept"]

Der R-Hat Wert liegt unter 1.01 also können wir beruhigt weitermachen. 

Wir würden jetzt gerne wissen, welche durchschnittliche Teilnahme Tendenz am wahrscheinlichsten ist, nachdem wir 18 von 20 mal ein Ja bekommen haben. Außerdem wäre es gut zu wissen welche anderen Anteile / Tendenzen noch glaubhaft sind. Dazu schauen wir uns den Posterior nun an.

In [ ]:
# Posterior auf Wahrscheinlichkeits-Skala transformieren
# (Intercept ist auf logit-Skala)

intercept_samples = results.posterior["Intercept"].values.flatten()
theta_samples = expit(intercept_samples)  # expit = logit^-1

print("\nGeschätzte Teilnahme-Wahrscheinlichkeit:")
print(f"  Mittelwert: {np.mean(theta_samples):.0%}")
print(
    f"  95% Credible Interval: [{np.percentile(theta_samples, 2.5):.0%}, "
    f"{np.percentile(theta_samples, 97.5):.0%}]"
)

Wir sehen, dass wir nach dem Erhalt der Daten in denen 90% der 20 Befragten mit Ja geantwortet haben, auch einen 70%-igen Anteil in der Gesamtheit der Vereinsmitglieder noch für plausibel halten (auch wenn wir 86% als den wahrscheinlichsten Wert ansehen). Dies verdeutlicht, dass wir nach der Befragung von nur 20 Mitgliedern noch erhebliche Unsicherheit haben.
Probiere selbst einmal aus wie sich das ändert, wenn wir bei 100 Befragten einen 90% Anteil an Ja-Antowrten bekommen hätten. (Hint: das Credible Intervall sollte auf ca. 82%-94% schrumpfen.)

## Bonus: Prior Visualisieren
Wir wollen einmal Daten aus dem Prior simulieren (prior predictive distribution) um den Standard prior, den Bambi verwendet besser zu verstehen. Das heißt wir schauen uns an welche Daten das Modell für wahrscheinlich hält, bevor wir ihm unseren Datensatz gezeigt haben.

In [ ]:
# Wir simulieren 10000 Datensätze der Größe 20 (Prior Predictive Distribution)
prior_pred = model.prior_predictive(draws=100000)

# Visualize prior predictive distribution
print("\nPrior Predictive Distribution:")
print("=" * 50)

# Extract prior predictive samples
prior_samples = prior_pred.prior_predictive["teilnahme"].values
# Sum over individuals to get total number of "yes" responses per sample
prior_n_ja = prior_samples.sum(axis=-1).flatten()

print("Prior predictive - erwartete Anzahl 'Ja'-Antworten:")
print(f"  Mittelwert: {np.mean(prior_n_ja):.1f} von {n_befragte}")
print(f"  Median: {np.median(prior_n_ja):.1f} von {n_befragte}")
print(
    f"  95% Prior Interval: [{np.percentile(prior_n_ja, 2.5):.1f}, "
    f"{np.percentile(prior_n_ja, 97.5):.1f}]"
)
print(f"  Beobachtet: {n_ja} von {n_befragte}\n")

In [ ]:
# Plot prior predictive distribution
fig_prior, ax_prior = plt.subplots(1, 1, figsize=(10, 6))

# Histogram of prior predictive
ax_prior.hist(
    prior_n_ja,
    bins=range(0, n_befragte + 2),
    density=True,
    alpha=0.6,
    edgecolor="black",
    color="skyblue",
    label="Prior Predictive",
    align="left",
)
ax_prior.axvline(
    n_ja, color="red", linestyle="--", linewidth=2, label=f"Beobachtete Daten: {n_ja}"
)
ax_prior.axvline(
    np.mean(prior_n_ja),
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"Prior Erwartung: {np.mean(prior_n_ja):.1f}",
)
ax_prior.set_xlabel('Anzahl "Ja"-Antworten (von {})'.format(n_befragte), fontsize=12)
ax_prior.set_ylabel("Wahrscheinlichkeitsdichte", fontsize=12)
ax_prior.set_title(
    "Prior Predictive Distribution\n(Was erwarten wir vor dem Sehen der Daten?)",
    fontsize=14,
    fontweight="bold",
)
ax_prior.legend(fontsize=11)
ax_prior.grid(alpha=0.3)
# plt.tight_layout()
plt.show()

Wir sehen, dass bambi einen prior gewählt hat, der kaum Einschränkungen
macht. Alle Anzahlen an wahrscheinlichen Ja Antworten sind etwa gleich
wahrscheinlich, außer die beiden Extreme von 0 oder 20 Ja Antworten.